# Lab 5 Practice: Using SVD for Latent Semantic Analysis (LSA)

Please complete worksheet_09.ipynb first. 


In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer

np.set_printoptions(precision=3, suppress=True)
rng = np.random.default_rng(0)


---
## Exercise 1 — Building a concept space from documents

LSA's whole point is: turn documents into a term-document matrix, run SVD on it, and use
the top singular vectors as a low-dimensional "concept space" where similar documents
end up near each other — even if they don't share many exact words.

Here are 8 short documents, four about **movies** and four about **cooking**:


In [ ]:
docs = [
    "the director praised the actor's performance in the new film",     # M1
    "critics loved the movie's lead actor and the emotional film",       # M2
    "the director shot the final scene of the movie in one take",        # M3
    "the actor and director discussed the movie's best scene",           # M4
    "the chef added fresh basil to the pasta recipe in the kitchen",     # C1
    "this recipe calls for butter garlic and a hot kitchen oven",        # C2
    "the chef seasoned the soup recipe with herbs in the kitchen",       # C3
    "a good kitchen and a simple recipe make the chef's best dish",      # C4
]
labels = ["M1", "M2", "M3", "M4", "C1", "C2", "C3", "C4"]


**(a) Prediction.** Before you run anything: these 8 documents don't share many exact
words within a topic (M1 talks about a "performance," M3 about a "take" — different
words!). Do you expect LSA's 2D concept space to still separate the movie docs from the
cooking docs into two visible clusters? Why might overlapping *context* words (not just
topic nouns) be enough for SVD to pick up on, even without a single word being shared by
all four documents in a group?

**(b) Coding.** Vectorize the documents with `CountVectorizer` (use `stop_words="english"`
so words like "the" and "and" don't swamp the real content words), run SVD on the
resulting term-document matrix, project the documents onto the top **k=2** singular
vectors to get a 2D concept coordinate for each document, and scatter-plot the result
with each point labeled and colored by topic.

**(c) Open-ended.** Look at the singular values. How many are clearly larger than the
rest, and does that match how many real topics are in this tiny corpus?


In [ ]:
# (b) TODO: vectorize, run SVD, project to k=2, and plot
cv = CountVectorizer(stop_words="english")
X = cv.fit_transform(docs).toarray().astype(float)

U, S, Vt = np.linalg.svd(X, full_matrices=False)
print("singular values:", S)

k = 2
Vk = ...  # top-k right singular vectors, terms x k
Z = ...   # projected documents, docs x k

colors = ["tab:blue"] * 4 + ["tab:orange"] * 4
plt.figure(figsize=(5, 5))
plt.scatter(Z[:, 0], Z[:, 1], c=colors)
for lab, z in zip(labels, Z):
    plt.annotate(lab, (z[0], z[1]))
plt.xlabel("concept 1")
plt.ylabel("concept 2")
plt.title("Documents projected into 2D concept space")
plt.axhline(0, color="lightgray", linewidth=0.5)
plt.axvline(0, color="lightgray", linewidth=0.5)
plt.show()


---
## Exercise 2 — Capturing synonyms that never co-occur

One of LSA's most useful (and surprising) properties: it can decide that two *different*
words are "close" in meaning, even if those two words never appear in the same document
together — as long as they each separately co-occur with the same other words.

Consider these three short documents:

- d1: "the actor memorized his lines for the new role"
- d2: "the performer memorized his lines for the new role"
- d3: "the chef added salt and pepper to the soup"

Notice: **"actor" and "performer" never appear in the same document.** Each only shows up
in its own sentence. But both of those sentences otherwise say almost exactly the same
thing.

**(a) Prediction.** In the raw term-document matrix, what is the cosine similarity
between the "actor" column and the "performer" column? What about "actor" and "chef"?

**(b) Coding.** Build the term-document matrix, run SVD, and compute the cosine
similarity between the **term vectors** for "actor" and "performer" (rows of $V$, i.e.
columns of $V^\top$) using only the **top k=1** singular direction. Compare to your
prediction in (a).

**(c) Open-ended.** Recompute that same similarity using *all* singular directions (no
rank reduction — $k$ = number of singular values). What happens, and what does that tell
you about *why* LSA needs to throw away the smaller singular directions in order to
surface this kind of synonymy?


In [ ]:
docs2 = [
    "the actor memorized his lines for the new role",
    "the performer memorized his lines for the new role",
    "the chef added salt and pepper to the soup",
]

cv2 = CountVectorizer(stop_words="english")
X2 = cv2.fit_transform(docs2).toarray().astype(float)
vocab2 = list(cv2.get_feature_names_out())
i_actor = vocab2.index("actor")
i_performer = vocab2.index("performer")
i_chef = vocab2.index("chef")

def cosine(u, v):
    return (u @ v) / (np.linalg.norm(u) * np.linalg.norm(v) + 1e-12)

# (a): raw co-occurrence based similarity (between term COLUMNS of X2)
print("raw cos(actor, performer):", cosine(X2[:, i_actor], X2[:, i_performer]))
print("raw cos(actor, chef):", cosine(X2[:, i_actor], X2[:, i_chef]))
print("raw cos(performer, chef):", cosine(X2[:, i_performer], X2[:, i_chef]))

# (b) TODO: SVD, then term vectors restricted to the top-1 singular direction
U2, S2, Vt2 = np.linalg.svd(X2, full_matrices=False)
k = 1
term_vecs_k1 = ...  # shape (n_terms, k) -- rows of V restricted to top-k columns
print("k=1 cos(actor, performer):", cosine(term_vecs_k1[i_actor], term_vecs_k1[i_performer]))
print("k=1 cos(actor, chef):", cosine(term_vecs_k1[i_actor], term_vecs_k1[i_chef]))
print("k=1 cos(performer, chef):", cosine(term_vecs_k1[i_performer], term_vecs_k1[i_chef]))

# (c) TODO: repeat using the full number of singular directions


---
## Exercise 3 — Semantic search: matching a query to documents

A classic use of LSA is semantic search: project a short query into the same concept
space as your document collection, then rank documents by how close they are to the
query — without needing the query to share exact words with the best-matching document.

We'll reuse the movie/cooking concept space from **Exercise 1** (`X`, `Vk`, `Z`, `cv`,
`labels`).

**(a) Prediction.** Here's a query: `"the lead actor shot a new scene"`. Without running
any code, which group of documents (movies or cooking) do you expect to rank highest?
Which specific document(s) do you expect to be the closest matches, and why?

**(b) Coding.** Turn the query into a count vector using the **same fitted**
`CountVectorizer` from Exercise 1 (`cv.transform(...)`, not a new vectorizer — the query
must land in the *same* vocabulary and the *same* concept space). Project it into the 2D
concept space using the *same* `Vk` (no need to recompute the SVD — this is exactly the
"folding in" trick from Exercise 1, just applied to a query instead of a new document).
Then rank all 8 documents by cosine similarity to the query in concept space.

**(c) Open-ended.** The query in (b) shares exact words with specific movie documents
("actor" is in M1/M4, "scene" is in M3/M4), so part of what's happening could just be
ordinary keyword overlap. Construct a second query using only words that **do** appear
somewhere in the movie documents' vocabulary, but where **no single movie document
contains the whole query** — i.e., the query's words are scattered across *different*
documents, so no document matches it directly. Check this with code (not just by eye).
Does concept-space search still rank it above the cooking documents? What does that tell
you about what it's actually matching on?


In [ ]:
query = "the lead actor shot a new scene"

# (b) TODO: use the SAME Vk from Exercise 1
q_vec = cv.transform([query]).toarray().astype(float)[0]
z_query = ...              # project into the same 2D concept space as Z

sims = [cosine(z_query, z) for z in Z]
for lab, s in sorted(zip(labels, sims), key=lambda t: -t[1]):
    print(lab, round(s, 3))


---
## Exercise 4 — Picking how many concepts to keep

LSA requires choosing $k$, the number of latent concepts to keep. We'll simulate a
document collection that actually has **3** real topics, each with its own vocabulary,
plus some generic words that show up everywhere regardless of topic — then see whether
the elbow method recovers $k=3$.


In [ ]:
topic_terms = {
    "space":   ["rocket", "orbit", "satellite", "astronaut", "mars", "launch"],
    "finance": ["stock", "market", "invest", "profit", "bank", "shares"],
    "cooking": ["recipe", "chef", "kitchen", "bake", "spice", "dinner"],
}
noise_terms = ["also", "really", "thing", "today", "said", "just"]

all_terms = sum(topic_terms.values(), []) + noise_terms
term_idx = {t: i for i, t in enumerate(all_terms)}
n_terms = len(all_terms)

n_docs_per_topic = 15
rows, doc_topics = [], []
for topic, words in topic_terms.items():
    topic_idxs = [term_idx[w] for w in words]
    noise_idxs = [term_idx[w] for w in noise_terms]
    for _ in range(n_docs_per_topic):
        row = np.zeros(n_terms)
        row[topic_idxs] = rng.poisson(lam=3.0, size=len(topic_idxs))
        row[noise_idxs] = rng.poisson(lam=2.0, size=len(noise_idxs))
        rows.append(row)
        doc_topics.append(topic)

A5 = np.array(rows)  # 45 documents x 24 terms, 3 real topics buried in generic words
print(A5.shape)


**(a) Coding.** Compute the SVD of `A5` and plot the first ~10 singular values in
descending order. Where's the elbow?

**(b) Prediction.** Based on the plot, what $k$ would you choose, and does it match the
3 real topics baked into the simulation?

**(c) Open-ended.** Now think about the two failure modes:
- If you picked $k=1$ here, what specifically would go wrong with the resulting
  document embeddings — in terms of grouping documents by *topic*?
- If you picked $k=20$ (deep into the flat part of the plot), what would go wrong
  instead?

You don't need code for (c); answer in terms of what you'd expect from the structure of
this simulated corpus (3 real topics + generic noise words shared by everyone).


In [ ]:
# (a) TODO: SVD of A5, then plot the first ~10 singular values
U5, S5, Vt5 = ...

plt.plot(range(1, 11), S5[:10], marker="o")
plt.xlabel("component index")
plt.ylabel("singular value")
plt.title("Singular value spectrum")
plt.show()
